# Databricks docs: RAG demo

Two ways to use the vector database the pipeline built:

- **Ask**: your question is embedded with the same model as the docs. Its 10 most
  similar rows are found in Qdrant and read back from `embeddings.sqlite`, and an
  OpenAI chat model, called through LiteLLM, answers from those 10 excerpts and
  cites them.
- **Search**: your query's 10 most similar rows, mapped back to their pages. No
  chat model is involved.

Before running it:

- The collection is loaded (`scripts/run_pipeline.sh`, or
  `databricks-docs-vectordb index run`).
- `.env` has `OPENAI_API_KEY` and `QDRANT_API_KEY`.
- The kernel is the project's `.venv`, and the notebook runs from the repo root,
  where `data/` and `.env` are.

Run the cells in order. Each question cell asks for its query when it runs, and can
be run again for another one.

## Setup

In [ ]:
from typing import NamedTuple

import litellm
from IPython.display import Markdown, display

from databricks_docs_vectordb.cli import load_provider_credentials
from databricks_docs_vectordb.embedding import provider
from databricks_docs_vectordb.embedding.config import Settings as EmbeddingSettings
from databricks_docs_vectordb.embedding.store import EmbeddingsStore
from databricks_docs_vectordb.indexing.config import Settings as IndexingSettings
from databricks_docs_vectordb.indexing.runner import configured_dimensions, unit_vector
from databricks_docs_vectordb.indexing.store import QdrantStore, collection_name
from databricks_docs_vectordb.parsing.chunker import own_texts
from databricks_docs_vectordb.parsing.config import Settings as ParsingSettings
from databricks_docs_vectordb.parsing.store import ChunkStore

CHAT_MODEL = "gpt-5.4-mini"
TOP_K = 10  # how many rows each question retrieves

embedding_settings = EmbeddingSettings.load()
settings = IndexingSettings.load()
parsing_settings = ParsingSettings.load()
if not settings.embeddings_path.exists():
    raise FileNotFoundError(
        f"no {settings.embeddings_path}: run the pipeline first, and this "
        "notebook from the repo root"
    )
# OPENAI_API_KEY into the environment for LiteLLM, the way `embed run` does it.
load_provider_credentials(embedding_settings)
litellm.suppress_debug_info = True

model = provider.configured_model_info(embedding_settings.model)
embeddings = EmbeddingsStore(settings.embeddings_path)
chunks = ChunkStore(parsing_settings.parsed_path)  # for Search's excerpts
qdrant = QdrantStore(settings)
dimensions = configured_dimensions(embeddings, model)
collection = collection_name(settings.collection_prefix, model, dimensions)
print(
    f"{collection}: {qdrant.count(collection):,} points. Questions are embedded "
    f"with {embedding_settings.model} and answered by {CHAT_MODEL}."
)

Retrieval, shared by both parts: embed the question, find the most similar rows in
Qdrant, and read each one back from `embeddings.sqlite` by its key, since the points
hold nothing else.

In [ ]:
class Passage(NamedTuple):
    """One retrieved row, read back from embeddings.sqlite."""

    score: float  # cosine similarity to the question: higher is closer
    chunk_id: str
    sub_index: int  # which slice of an oversized chunk: 0 for nearly all
    url: str
    title: str
    heading: str  # "" for text above the page's first heading
    text: str  # the embedded text, without its title, heading and section lines


async def retrieve(question: str, k: int = TOP_K) -> list[Passage]:
    """The k stored rows most similar to `question`, best first."""
    batch = await provider.embed([question], model=embedding_settings.model)
    vector = batch.vectors[0]
    if settings.normalize:  # as the stored vectors are
        vector = unit_vector(vector)
    passages = []
    for hit in qdrant.search(collection, vector, limit=k):
        chunk_id, sub_index = hit.payload["chunk_id"], hit.payload["sub_index"]
        row = embeddings.lookup(chunk_id, sub_index)
        if row is None:  # a point with no row; `index verify` reports these
            continue
        # The first four lines are the title, heading and section, then a blank.
        text = row.embedded_text.split("\n", 4)[-1]
        passages.append(
            Passage(
                hit.score,
                chunk_id,
                sub_index,
                row.url,
                row.doc_title,
                row.heading_path,
                text,
            )
        )
    return passages


def label(passage: Passage) -> str:
    """The page title, and the heading within it when there is one."""
    if passage.heading:
        return f"{passage.title} > {passage.heading}"
    return passage.title

## Ask a question (RAG)

The chat model gets your question and the 10 excerpts, numbered, and is told to
answer from them alone and to cite them.

In [ ]:
SYSTEM_PROMPT = """\
You answer questions about Databricks using only the numbered excerpts from the \
Databricks documentation that come with each question. Cite the excerpts you use \
by number, like [2]. If they don't answer the question, say so instead of guessing."""


def ask(question: str, passages: list[Passage]) -> str:
    """The chat model's answer to `question`, from `passages` alone."""
    excerpts = "\n\n".join(
        f"[{n}] {label(p)}\n{p.url}\n{p.text}" for n, p in enumerate(passages, 1)
    )
    response = litellm.completion(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": f"Excerpts:\n\n{excerpts}\n\nQuestion: {question}",
            },
        ],
    )
    return response.choices[0].message.content

In [ ]:
question = input("Ask a question about Databricks: ")

In [ ]:
passages = await retrieve(question)
answer = ask(question, passages)


def source(n: int, p: Passage) -> str:
    text = label(p).replace("[", r"\[").replace("]", r"\]")
    return f"{n}. [{text}]({p.url}) ({p.score:.3f})"


sources = "\n".join(source(n, p) for n, p in enumerate(passages, 1))
display(Markdown(f"{answer}\n\n**Sources**\n\n{sources}"))

## Search: the top 10 rows, mapped back to their pages

Your query's 10 most similar rows, best first: each one's similarity, page, heading
and the start of what it says. Each chunk begins with the end of the chunk before it
on the page, so that overlap is taken off first. Then the pages they come from.

In [ ]:
search_query = input("Search the Databricks docs: ")

In [ ]:
def own_start(p: Passage, length: int = 160) -> str:
    """The start of what `p`'s chunk itself says, without the overlap the chunker
    repeats from the chunk before it. A later slice of an oversized chunk already
    starts mid-chunk, so it is shown as it is."""
    text = p.text
    if p.sub_index == 0:
        page = chunks.get_chunks(p.url)
        owns = dict(zip((c.chunk_id for c in page), own_texts(page, parsing_settings)))
        text = owns.get(p.chunk_id, text)
    return " ".join(text.split())[:length]


results = await retrieve(search_query)
for n, p in enumerate(results, 1):
    print(f"{n:>2}. {p.score:.3f}  {label(p)}")
    print(f"           {p.url}")
    print(f"           {own_start(p)}...")
    print()

ranks: dict[str, list[int]] = {}  # each page's hits, in rank order
for n, p in enumerate(results, 1):
    ranks.setdefault(p.url, []).append(n)
print(f"{len(ranks)} page(s) among the top {len(results)}:")
for url, hits in ranks.items():
    print(f"  {url}  (#{', #'.join(map(str, hits))})")